# Question 2: Chunking for RAG

Follow the document → overlapping chunks approach from the [foundation RAG notebook](https://github.com/alexeygrigorev/ai-engineering-buildcamp-code/blob/main/01-foundation/02-rag/rag.ipynb).

For this homework, the content is a **list of non-empty lines**, so `size=100` and `step=50` measure lines rather than characters. Use `gitsource.chunk_documents` to preserve each book’s source in its chunks.

Run the cells in order using the project’s Python environment. `gitsource` is already a project dependency; if missing, run `uv add gitsource` from the repository root.

In [ ]:
from pathlib import Path
from gitsource import chunk_documents

# Support running from the repository root or homework/.
project_root = Path.cwd()
if not (project_root / "books_text").is_dir():
    project_root = project_root.parent

books_text_dir = project_root / "books_text"
markdown_files = sorted(books_text_dir.glob("*.md"))
if not markdown_files:
    raise FileNotFoundError(f"No Markdown books found in {books_text_dir}")

## Prepare the documents

Remove lines for which `line.strip()` is empty. Keep the text of all remaining lines unchanged.

In [ ]:
documents = []

for markdown_path in markdown_files:
    lines = markdown_path.read_text(encoding="utf-8").splitlines()
    non_empty_lines = [line for line in lines if line.strip()]
    documents.append({
        "source": markdown_path.name,
        "content": non_empty_lines,
    })

print(f"Prepared {len(documents)} books.")
for document in documents:
    print(f"{document['source']}: {len(document['content']):,} non-empty lines")

## Create overlapping chunks

Each window contains up to 100 lines and advances by 50 lines. The package handles the final window and retains the `source` and starting position.

In [ ]:
document_chunks = chunk_documents(documents, size=100, step=50)
print(f"Total chunks across all books: {len(document_chunks):,}")

## Answer: Think Python, second edition

In [ ]:
think_python_source = "thinkpython2.md"
if not any(document["source"] == think_python_source for document in documents):
    raise FileNotFoundError(f"{think_python_source} was not found in {books_text_dir}")

think_python_chunks = [
    chunk for chunk in document_chunks
    if chunk["source"] == think_python_source
]

print(f"Think Python (2nd edition): {len(think_python_chunks)} chunks")